# CYR-GPU-013 / R1B — replicated vocabulary response curve

Use **Runtime → Change runtime type → T4 GPU**, then **Run all**. The notebook fails closed unless the frozen executable, unit tests, CUDA checks, and two-curve runtime projection all pass before scientific training.


In [ ]:
# CELL 0 — FREEZE / VERIFY / TEST / CALIBRATE / RESOLVE
import json, os, subprocess, sys
from pathlib import Path
REPO=Path('/content/An-Ra-the-new-AGI-r1b')
BRANCH='cymek-500m-readiness'
REMOTE='https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
if not REPO.exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REMOTE,str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','-q',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard',f'origin/{BRANCH}'],check=True)
P=REPO/'docs/cymek/experiments/CYR-GPU-013-R1B/PREREGISTRATION.json'
R=REPO/'docs/cymek/experiments/CYR-GPU-013-R1B/RUN_READINESS.json'
if not P.exists() or not R.exists(): raise RuntimeError('R1B is not frozen/readiness-bound')
PREREG=json.loads(P.read_text()); READINESS=json.loads(R.read_text())
if READINESS.get('ready_for_operator_colab_gpu_run') is not True: raise RuntimeError('R1B readiness is not green')
if READINESS['executable_sha'] != PREREG['executable_sha']: raise RuntimeError('R1B readiness/prereg SHA mismatch')
EXECUTABLE_SHA=PREREG['executable_sha']
Path('/content/CYR-GPU-013-R1B-PREREGISTRATION.json').write_text(json.dumps(PREREG,indent=2,sort_keys=True))
subprocess.run(['git','-C',str(REPO),'checkout','-q',EXECUTABLE_SHA],check=True)
HEAD=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip(); assert HEAD==EXECUTABLE_SHA
for relative, expected in PREREG['executable_blobs'].items():
    actual=subprocess.check_output(['git','-C',str(REPO),'hash-object',relative],text=True).strip()
    assert actual==expected, f'blob mismatch: {relative}'
print('R1B frozen executable verified:',HEAD)
subprocess.run([sys.executable,'-m','pip','install','-q','tokenizers','pytest','numpy'],check=True)
os.chdir(REPO); sys.path.insert(0,str(REPO))
subprocess.run([sys.executable,'-m','py_compile','v5_experiments/cyr_gpu013_r1b.py','anra_v5/cyr_gpu013_r1b_run.py'],check=True)
subprocess.run([sys.executable,'-m','pytest','tests/test_v5_cyr_gpu013_r1b.py','tests/test_v5_cyr_gpu012_r1.py','tests/test_v5_cyr_gpu011.py','tests/test_v5_cyr_gpu011_entry.py','-q'],check=True)
import torch
if not torch.cuda.is_available(): raise RuntimeError('R1B requires a Colab CUDA GPU')
DEVICE=torch.device('cuda'); print('GPU:',torch.cuda.get_device_name(0),'VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
from v5_experiments import cyr_gpu011 as base
from anra_v5.cyr_gpu013_r1b_run import calibrate_all
from v5_experiments.cyr_gpu013_r1b import resolve_from_calibrations
data=base.load_ark002b_manifest(REPO/'docs/cymek/experiments/CYR-GPU-011/ARK002B_TASK_MANIFEST.json')
assert data['source_split_sha256']==PREREG['data']['split_sha256']; assert data['source_blob_sha']==PREREG['data']['manifest_blob_sha']
CALIBRATIONS=calibrate_all(data=data,torch=torch,device=DEVICE)
Path('/content/CYR-GPU-013-R1B-CALIBRATIONS.json').write_text(json.dumps(CALIBRATIONS,indent=2,sort_keys=True))
for k,v in sorted(CALIBRATIONS.items()): print(k,v.get('status'),'updates/s=',round(float(v.get('training_updates_per_sec',0)),3),'peak_GB=',round(float(v.get('peak_vram_gb',0)),2))
RESOLVED=resolve_from_calibrations(CALIBRATIONS)
Path('/content/CYR-GPU-013-R1B-RESOLVED.json').write_text(json.dumps(RESOLVED,indent=2,sort_keys=True))
print(json.dumps(RESOLVED,indent=2)); print('R1B PREEXECUTION GATE: PASS')


In [ ]:
# CELL 1 — RUN / DURABLE DRIVE OUTPUT
import json, sys
from pathlib import Path
import torch
from google.colab import drive
drive.mount('/content/drive')
REPO=Path('/content/An-Ra-the-new-AGI-r1b'); sys.path.insert(0,str(REPO))
PREREG=json.loads(Path('/content/CYR-GPU-013-R1B-PREREGISTRATION.json').read_text())
CALIBRATIONS=json.loads(Path('/content/CYR-GPU-013-R1B-CALIBRATIONS.json').read_text())
RESOLVED=json.loads(Path('/content/CYR-GPU-013-R1B-RESOLVED.json').read_text())
OUT=Path('/content/drive/MyDrive/CYMEK/CYR-GPU-013-R1B'); OUT.mkdir(parents=True,exist_ok=True)
from anra_v5.cyr_gpu013_r1b_run import run_campaign
campaign=run_campaign(repo=REPO,out=OUT,preregistration=PREREG,resolved=RESOLVED,calibrations=CALIBRATIONS,torch=torch,device=torch.device('cuda'),progress=lambda m: print(m,flush=True))
print('STATUS:',campaign['status']); print('COMPLETE CURVES:',campaign.get('complete_curves')); print('VERDICT:',campaign.get('decision',{}).get('verdict')); print('BUNDLE:',campaign['bundle']['path'])


In [ ]:
# CELL 2 — VERIFY / DOWNLOAD
import hashlib,json
from pathlib import Path
from google.colab import files
OUT=Path('/content/drive/MyDrive/CYMEK/CYR-GPU-013-R1B')
receipt=json.loads((OUT/'campaign_receipt.json').read_text())
bundle=Path(receipt['bundle']['path']); assert bundle.exists()
actual=hashlib.sha256(bundle.read_bytes()).hexdigest(); assert actual==receipt['bundle']['sha256']
print('verified:',bundle.name,actual); print('status:',receipt['status']); print('verdict:',receipt.get('decision',{}).get('verdict'))
for row in receipt.get('decision',{}).get('per_seed',[]): print(row)
files.download(str(bundle))
